In [ ]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw
from ngsolve.bem import *


**keys**: EM Scattering, Brakhage-Werner formulation, Curl-Curl equation, Dirichlet data

# Brakhage Werner for EM

EM-Scattering at PEC: 

$$ \left\{ \begin{array}{rcl l} \mathbf{curl} \, \mathbf{curl}\, \boldsymbol E - \kappa^2 \, \boldsymbol E &=& \boldsymbol 0, \quad &\textnormal{in } \Omega^c \subset \mathbb R^3\,,\\ \gamma_R \,\boldsymbol E &=& \boldsymbol m, \quad & \textnormal{on }\Gamma \\ \left\| \mathbf{curl} \, \boldsymbol E( x) - i\,\omega\,\epsilon \, \boldsymbol E( x)\right\| &=& \mathcal O\left( \displaystyle \frac{1}{\| x\|^2}\right), &\textnormal{for} \; \|x\| \to \infty\,.\end{array} \right. $$ 

Combined Field Integral Equation Ansatz (stable wrt eigenvalues):


$$
\boldsymbol E = i\,\mathrm{SL}(\boldsymbol j) - \mathrm{DL}(\boldsymbol j)\,,
$$

- $\mathrm{SL}$ Maxwell Single Layer Potential
- $\mathrm{DL}$ Maxwell Double Layer Potential

Combined Field Integral Equation for $\boldsymbol j$

$$
    \left( K + i \, V + K - \tfrac12\right) \boldsymbol j = \boldsymbol m \qquad \text{on} \, \Gamma \,,
$$


- $\mathrm{V}$ Maxwell Single Layer Operator
- $\mathrm{K}$ Maxwell Double Layer Operator 


In [ ]:
face = Glue(Sphere((0,0,0),1).faces)
mesh = Mesh(OCCGeometry(face).GenerateMesh(maxh=0.2)).Curve(4)
Draw (mesh);

visplane = WorkPlane(Axes((0,0,0), Z, X)).RectangleC(5,5).Face()
vismesh = Mesh(OCCGeometry(visplane).GenerateMesh(maxh=0.2))
Draw (vismesh);

In [ ]:
kappa = 5*pi
fes = HDivSurface(mesh, order=4, complex=True)
u,v = fes.TnT()

d = CF((0,-kappa,0))
E_inc = exp(1j*d*CF((x,y,z))) * CF((0,0,1))

Draw (E_inc[2], vismesh, animate_complex=True, order=3);

In [ ]:
n = specialcf.normal(3)
bio = 2
with TaskManager():
    jV = 1j*kappa*(HelmholtzSL(u.Trace()*ds(bonus_intorder=bio), kappa)*v.Trace()*ds(bonus_intorder=bio)).mat \
            - 1/(1j*kappa)*(HelmholtzSL(div(u.Trace())*ds(bonus_intorder=bio), kappa)*div(v.Trace())*ds(bonus_intorder=bio)).mat

    K = (HelmholtzDL(Cross(-n,u.Trace())*ds(bonus_intorder=bio), kappa)*v.Trace()*ds(bonus_intorder=4)).mat

    M = BilinearForm(u.Trace()*v.Trace()*ds).Assemble().mat

lhs = jV + K - 0.5*M


In [ ]:
gf = GridFunction(fes)
rhs = LinearForm(E_inc*v.Trace()*ds(bonus_intorder=bio)).Assemble()
pre = BilinearForm(u.Trace()*v.Trace()*ds).Assemble().mat.Inverse(inverse="sparsecholesky")

with TaskManager():
    gf.vec[:] = solvers.GMRes(A=lhs, b=rhs.vec, pre=pre, maxsteps=400, tol=1e-8)


In [ ]:
Draw (gf[2], mesh, animate_complex=True)

In [ ]:
SL = 1j*kappa*HelmholtzSL(u.Trace()*ds(bonus_intorder=bio), kappa)(gf) \
        -1/(1j*kappa)*grad(HelmholtzSL(div(u.Trace())*ds(bonus_intorder=bio), kappa)(gf))
DL = HelmholtzDL(Cross(-n,u.Trace())*ds(bonus_intorder=bio), kappa)(gf)


gfSL = GridFunction(VectorH1(vismesh, order=3, complex=True))
gfDL = GridFunction(VectorH1(vismesh, order=3, complex=True))

with TaskManager():
    gfSL.Set(SL, definedon=vismesh.Boundaries(".*"))
    gfDL.Set(DL, definedon=vismesh.Boundaries(".*"))

In [ ]:
E_scat=gfSL+gfDL
Draw (E_inc[2], vismesh, animate_complex=True, min=-1, max=1)
Draw (E_scat[2], vismesh, animate_complex=True, min=-1, max=1)

Draw ( (E_inc-E_scat)[2], vismesh, animate_complex=True, min=-1, max=1)